# 01 - Initial Cleaning

Loads the raw SF registered business dataset from Dropbox, cleans column names, converts locations to a geodataframe, parses dates, and clips to the SF boundary. Filters to businesses with an end date of 2019 or later (or none), then splits into openings and closings tagged by year and status. Exports the combined result to `ALL_openings_closings.parquet`.

<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/processing_notebooks/01_initial_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from shapely.geometry import LineString
import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
url = 'https://www.dropbox.com/scl/fi/wsia6aakay4x22b8hnf9l/rbl_sf.csv?rlkey=giuvlx1efcog4vre2dovozz7h&st=lu0ab5l7&dl=1'
gdf = pd.read_csv(url)

In [25]:
###renaming columns
new_names = gdf.columns.str.strip().str.replace(' #', '_num').str.replace(' - ', '_').str.replace(' ', '_').str.lower()
gdf.columns = new_names
gdf.columns

Index(['uniqueid', 'business_account_number', 'location_id', 'ownership_name',
       'dba_name', 'street_address', 'city', 'state', 'source_zipcode',
       'business_start_date', 'business_end_date', 'location_start_date',
       'location_end_date', 'administratively_closed', 'mail_address',
       'mail_city', 'mail_state', 'mail_zipcode', 'naics_code',
       'naics_code_description', 'naics_code_descriptions_list', 'lic_code',
       'lic_code_description', 'lic_code_descriptions_list', 'parking_tax',
       'transient_occupancy_tax', 'business_location', 'business_corridor',
       'neighborhoods_analysis_boundaries', 'supervisor_district',
       'community_benefit_district', 'data_as_of', 'data_loaded_at'],
      dtype='object')

In [26]:
from shapely import wkt

gdf = gdf[gdf["business_location"].notna()].copy()

# Converting string to geometry
gdf["geometry"] = gdf["business_location"].apply(wkt.loads)

#Right now it's a df so I'm making it a gdf
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")

In [27]:
##setting crs and correcting datetimes

gdf['business_start_date'] = pd.to_datetime(gdf['business_start_date'], format = '%m/%d/%Y', errors = 'coerce')
gdf['business_end_date'] = pd.to_datetime(gdf['business_end_date'], format = '%m/%d/%Y', errors = 'coerce')
gdf['location_start_date'] = pd.to_datetime(gdf['location_start_date'], format = '%m/%d/%Y', errors = 'coerce')
gdf['location_end_date'] = pd.to_datetime(gdf['location_end_date'], format = '%m/%d/%Y', errors = 'coerce')


In [28]:
# adding another plot and importing a shpfile of SF so we can see where the points are in SF



# Importing SF geometry
# URL for 2025 TIGER/Line Place boundaries - info here on
## how to use: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
url = "https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_06_place.zip"

places = gpd.read_file(url)

# Filtered to SF
sf_poly = places[
    (places["NAME"] == "San Francisco") &
    (places["STATEFP"] == "06")   # 06 = California
]


# eproject to same as our gdf
sf_poly = sf_poly.to_crs(epsg=4326)

# keeping just the points in sf bc there are a lot outside
gdf_sf = gpd.sjoin(gdf, sf_poly, predicate="within") ##joined with gdf_hascode_sf instead, so we only have coded businesses -Sean

In [29]:
cols_to_keep = [
    'uniqueid', 'business_account_number', 'location_id', 'ownership_name',
    'dba_name', 'business_start_date', 'business_end_date', 'location_start_date',
    'location_end_date', 'naics_code','naics_code_description', 'lic_code',
    'lic_code_description', 'business_corridor','neighborhoods_analysis_boundaries',
    'geometry', 'administratively_closed'
]

gdf_sf = gdf_sf[cols_to_keep].copy()

#gdf_sf = gdf_sf[gdf_sf['administratively_closed'] != '***Administratively Closed']


In [30]:
#filter for location end dates 2016 onwards and businesses without end dates (still open)

filtered_gdf = gdf_sf[
    (gdf_sf['location_end_date'].dt.year >= 2019) | (gdf_sf['location_end_date'].isna())
]

###Creating opening and closing gdfs

In [31]:

# Making a year column

# converting the start and end dates to years using datetime
filtered_gdf['location_start_date'] = pd.to_datetime(filtered_gdf['location_start_date'], errors='coerce')
filtered_gdf['location_end_date'] = pd.to_datetime(filtered_gdf['location_end_date'], errors='coerce')

# making df for business openings
openings = filtered_gdf.copy()
openings["year"] = openings["location_start_date"].dt.year
openings["status"] = "opened"

## need to clip openings to 2016-2025 start dates
openings = openings[(openings["year"] >= 2019) & (openings["year"] <= 2025)]

# making df for business closings
closings = filtered_gdf[filtered_gdf["location_end_date"].notna()].copy()
closings["year"] = closings["location_end_date"].dt.year
closings["status"] = "closed"

## need to clip openings to 2016-2025 end dates
closings = closings[(closings["year"] >= 2019) & (closings["year"] <= 2025)]

gdf_open_close = pd.concat([openings, closings])


/usr/local/lib/python3.12/dist-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.12/dist-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)



##Exporting



In [32]:
#exporting to parquet
gdf_open_close.to_parquet('ALL_openings_closings.parquet')
from google.colab import files
files.download('ALL_openings_closings.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>